### Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [34]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY") or ""
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY") or ""
os.environ["LANGSMITH_TRACING"]=os.getenv("LANGSMITH_TRACING") or ""
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT") or ""

In [35]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="stepfun/step-3.5-flash:free")
model

ChatOpenRouter(profile={'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<openrouter.sdk.OpenRouter object at 0x0000023CC9E516E0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='stepfun/step-3.5-flash:free', model_kwargs={})

In [36]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, my name is Shashank and I am an AI Engineer")])

AIMessage(content="Hello Shashank! Nice to meet you. As an AI Engineer, you must be working on some fascinating projects. How are you finding it? Is there anything specific you'd like to discuss or explore today?", additional_kwargs={'reasoning_content': 'Hmm, the user introduced themselves as Shashank, an AI Engineer. This is a straightforward self-introduction with clear context. \n\nSince Shashank mentioned his profession, I can reasonably assume he\'s interested in technical or AI-related topics. A simple greeting with an open-ended question about his work would be appropriate here—it acknowledges his introduction while inviting him to elaborate. \n\nI should keep it friendly but professional, matching his tone. No need for excessive formality or small talk since he likely wants to get to the point. The response should feel natural, like a colleague starting a conversation. \n\nAvoiding assumptions about his specific projects or expertise—just leaving the door open for him to guide

In [50]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi, my name is Shashank and I am an AI Engineer"),
        AIMessage(content="Hello Shashank! Nice to meet you. As an AI Engineer, you must be working on some fascinating projects. How are you finding it? Is there anything specific you'd like to discuss or explore today?"),
        HumanMessage(content="Hey what's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **Shashank**, and you\'re an **AI Engineer**. 👋  \nGreat to "meet" you again!', additional_kwargs={'reasoning_content': 'Okay, the user asked, "Hey what\'s my name and what do I do?" Let me check the conversation history to understand the context.\n\nIn the previous interaction, the user introduced themselves as Shashank, an AI Engineer. So they already provided their name and profession. \n\nHmm, maybe they\'re testing if I remember correctly or just confirming. Since it\'s a straightforward recall, I should confirm both details accurately based on what they stated earlier.\n\nNo need for extra information, just repeat what they said to show attentiveness. Keep it simple and friendly to match their casual tone.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'Okay, the user asked, "Hey what\'s my name and what do I do?" Let me check the conversation history to understand the context.\n\nIn the previous interaction, the user introduced themsel

#### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [51]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history (session_id : str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [52]:
config = {"configurable": {"session_id": "chat1"}}

In [53]:
from typing import cast
from langchain_core.runnables import RunnableConfig

res = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Shashank and I am an AI Engineer"),
    ],
    config=cast(RunnableConfig, config),
)

In [54]:
res.content

"Hello, Shashank! 👋 It's great to meet you — an AI Engineer is a fascinating and impactful role.  \n\nHow can I assist you today? Whether it's about AI concepts, coding help, project brainstorming, industry trends, or anything else, I'm here to help."

In [55]:
with_message_history.invoke(
    [
        HumanMessage(content="What's my name?"),
    ],
    config=cast(RunnableConfig, config),
)

AIMessage(content="Your name is **Shashank**. 😊  \nNice to meet you again! Is there anything specific you'd like to discuss or work on today?", additional_kwargs={'reasoning_content': 'Hmm, the user just asked "What\'s my name?" right after introducing themselves as Shashank in the previous message. This seems like a simple recall test since the context is fresh in the conversation history. \n\nThe user likely wants to confirm if I\'m paying attention or just testing basic memory functionality. Since they explicitly stated their name earlier, I can directly retrieve it from the context without needing any clarification. \n\nShould keep it straightforward and friendly—no need for extra commentary since the question is literal. Just state the name clearly and maybe add a lighthearted touch since the context is casual. \n\nAh, and no ulterior motive seems present here; it\'s probably just a quick check. No need to overcomplicate the response.', 'reasoning_details': [{'type': 'reasoning.te

#### Change the config (new session)

In [56]:
config1 = {"configurable": {"session_id": "chat2"}}
res = with_message_history.invoke(
    [
        HumanMessage(content="What's my name?"),
    ],
    config=cast(RunnableConfig, config1),
)
res.content

"I don't have access to personal information about you unless you've shared it with me in our current conversation. Since this is a new session, I don’t know your name. 😊\n\nIf you’d like, you can tell me your name, and I’ll be happy to use it! Otherwise, I’m here to help with any questions or tasks you have."

In [57]:
res = with_message_history.invoke(
    [
        HumanMessage(content="My name is John"),
    ],
    config=cast(RunnableConfig, config1),
)
res.content

'Nice to meet you, John! 👋  \nI’ll remember your name for the rest of our conversation. How can I help you today?'

In [58]:
res = with_message_history.invoke(
    [
        HumanMessage(content="What's my name?"),
    ],
    config=cast(RunnableConfig, config1),
)
res.content

'Your name is **John**. 😊  \nI remember—nice to see you again!'

#### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [59]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "you're a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [73]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [74]:
config = {"configurable": {"session_id": "chat3"}}
with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi my name is Shashank")], 
    },
    config = cast(RunnableConfig, config)
)

AIMessage(content='Hi Shashank! 😊  \nNice to see you again. I’ve got your name saved — it’s good to reconnect. How can I help you today?', additional_kwargs={'reasoning_content': 'Hmm, the user just repeated the introduction they gave earlier. They already told me their name is Shashank in the previous conversation, and I’ve been using it since then. \n\nThey might be testing if I remember, or maybe just greeting me again. Either way, I should acknowledge it warmly but also confirm I already know their name to avoid redundancy. \n\nSince the context is clear and consistent, I’ll greet them back, repeat their name to reinforce I remember, and leave the door open for them to ask for help. Keeping it friendly and efficient.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'Hmm, the user just repeated the introduction they gave earlier. They already told me their name is Shashank in the previous conversation, and I’ve been using it since then. \n\nThey might be testing if I remem

In [85]:
from langchain_classic.schema import StrOutputParser
parser = StrOutputParser()

res = with_message_history.invoke(
    [
        HumanMessage(content="What's my name?"),
    ],
    config=cast(RunnableConfig, config),
)
print(parser.parse(res.content))

Your name is **Shashank** — you told me earlier in our conversation! 😊  
Is there something specific you'd like help with today?


In [101]:
# Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model | parser

In [105]:
res = chain.invoke({"messages":[HumanMessage(content="Hi My name is Shashank")],"language":"Hindi"})
res

'नमस्ते शशांक! आपका स्वागत है। मैं आपकी किस तरह सहायता कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [109]:
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages",
)

In [111]:
config = {"configurable": {"session_id": "chat4"}}

res = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi, I'm Shashank")], 
        "language": "Hindi"
    },
    config=cast(RunnableConfig, config)
)
res

'नमस्ते शशांक! फिर से मिलकर खुशी हुई 🌟  \nअब आप बताएं, आज मैं आपकी किस तरह से मदद कर सकता हूँ? कुछ पूछना है, कोई सवाल है, या फिर बस बात करना चाहते हैं — मैं यहाँ हूँ!'

In [112]:
res = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what's my name")], 
        "language": "Hindi"
    },
    config=cast(RunnableConfig, config)
)
res

'आपका नाम शशांक है! 😊  \nमैंने पिछली बार आपके परिचय में यह सुना था। क्या मैं आपकी किसी और मदद कर सकता हूँ?'

#### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.  
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [114]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy='last',
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [119]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("messages") | trimmer
    ) | prompt | model | parser
)

res = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do i like?")],
        "language": "English"
    }
)
res

'You mentioned that you like **vanilla ice cream**! 😊🍦'

In [120]:
res = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What math problem did i ask?")],
        "language": "English"
    }
)
res

'You asked: "whats 2 + 2"'

In [125]:
# Wrapping this in Message History
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable": {"session_id": "chat5"}}

In [127]:
res = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What is my name?")],
        "language": "English"
    },
    config=cast(RunnableConfig, config)
)
res

'Your name is Bob.'